# 第二阶段：逻辑回归与 Pipeline

本 Notebook 的目标是把第一阶段清洗后的客户数据交给机器学习模型。运行顺序是：
1. 读取并清洗数据，准备特征 X 和标签 y；
2. 划分训练集和测试集；
3. 用 ColumnTransformer 定义数值标准化和类别独热编码；
4. 用 Pipeline 把预处理和逻辑回归连接起来；
5. 在测试集上预测类别和流失概率；
6. 计算 ROC-AUC，并比较不同分类阈值；
7. 保存完整 Pipeline，之后由 Streamlit 应用加载。

注意：训练集用于学习规则，测试集只用于检验模型；不要把测试集信息用于训练。

## 1. 数据读取与特征准备


In [5]:
# 数据读取与基础检查
import pandas as pd

# 读取项目使用的原始电信客户数据
df = pd.read_csv('telco_customer_churn.csv')

# 查看前 5 行，快速确认字段名称和数据内容
data_head = df.head()
print(data_head)

# 查看数据集维度，返回值依次为（行数，列数）
data_shape = df.shape
print(data_shape)

# 查看字段类型和非空情况；df.info() 会直接打印结果，不需要再次 print
df.info()

# 查看所有字段名
data_columns = df.columns
print(data_columns)

# 每列唯一值数量
print(df.nunique())

# 检查 TotalCharges 中去掉首尾空格后是否为空字符串
total_charges_clean = df["TotalCharges"].astype(str).str.strip()
is_blank_total_charges = total_charges_clean.eq("")
print("空字符串数量：", is_blank_total_charges.sum())

# 查看指定列取值
print(df["Churn"].unique())
print(df["Contract"].unique())
print(df["TechSupport"].unique())


# 筛选 TotalCharges 为空的异常记录，并只查看关键字段
print(df[is_blank_total_charges][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

# 创建清洗副本，后续只修改副本以保留原始数据
df_clean = df.copy()
# tenure 为 0 的新客户尚未产生累计费用，因此把对应的空 TotalCharges 设为 0
df_clean.loc[is_blank_total_charges, 'TotalCharges'] = 0
# 将 TotalCharges 字段转换为数值，供模型训练使用
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'])
# 验证清洗结果：查看字段类型、缺失值和原空值记录
df_clean.info()
print(df_clean["TotalCharges"].isnull().sum())
print(df_clean.loc[is_blank_total_charges,["customerID", "tenure", "TotalCharges"]])



# 明确 X（输入特征）和 y（目标标签），并区分数值特征与类别特征
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['gender','Partner','Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']
# 类别特征会在 Pipeline 中通过 OneHotEncoder 转换为模型可用的数值列
# 合并两个列表，固定训练和预测时使用的字段范围与顺序
feature_columns = numeric_features + categorical_features
X = df_clean[feature_columns]
# 将 Churn 的 Yes 转换为 1（流失），No 转换为 0（未流失）
y = df_clean['Churn'].map({'Yes':1,'No':0})

print(X.shape)
print(X.columns)
print(y.value_counts())


# 按 8:2 划分训练集和测试集；stratify=y 保持两边流失比例接近
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())


# 定义特征预处理规则
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        # num 和 cat 是两个转换步骤的名称，可用于后续定位和检查。
        ("num", StandardScaler(), numeric_features),
        # handle_unknown="ignore" 可避免预测时出现新类别而直接报错。
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)
# 预处理将在下面的 Pipeline 中自动执行





# 模型训练、概率预测和阈值判断统一放在下面的 Pipeline 单元格中完成


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

## 2. 构建并训练 Pipeline


In [6]:
# Pipeline 结构：把预处理和分类器串成一条可重复执行的流程。
# 训练时先让 preprocessor 学习规则，再让 classifier 学习标签；预测时按相同顺序执行。
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# preprocessor 负责转换原始字段，classifier 负责根据转换后的特征预测流失。
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

# fit 会依次拟合预处理器和逻辑回归模型。
pipeline.fit(X_train, y_train)

# predict 输出 0/1 类别；predict_proba 输出属于每个类别的概率。
y_pred_pipeline = pipeline.predict(X_test)
y_proba_pipeline = pipeline.predict_proba(X_test)[:, 1]


print(len(y_pred_pipeline))
print(len(y_proba_pipeline))
print(type(X_train))
print(type(X_test))
print(type(y_train))
print(type(preprocessor))

1409
1409
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>
<class 'sklearn.compose._column_transformer.ColumnTransformer'>


## 3. ROC-AUC 评价


In [7]:
# roc_auc_score(真实标签, 流失概率) 用于计算 ROC-AUC。
# ROC-AUC 衡量模型把流失客户排在未流失客户前面的整体能力，
# 它使用连续概率进行计算，因此不依赖某一个固定分类阈值。
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(
    y_test,
    y_proba_pipeline
)

print(f"ROC-AUC：{roc_auc:.3f}")

ROC-AUC：0.842


## 4. 分类阈值实验


In [9]:
# 导入阈值实验需要的评价函数。它们的基本用法都是：
# 指标函数(真实标签, 预测类别)。
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
# 用列表保存准备比较的四个阈值，for 循环会依次测试每一个阈值。
thresholds = [0.50, 0.45, 0.40, 0.35]
for threshold in thresholds:
    # 1. 根据当前阈值生成预测类别。
    # >= threshold 先得到 True/False，astype(int) 再把它们转换为 1/0。
    y_pred_threshold = (y_proba_pipeline >= threshold).astype(int)

    # 2. 计算三个指标：Precision 关注误报，Recall 关注漏报，F1 综合两者。
    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    # 3. 计算混淆矩阵，默认结构是 [[TN, FP], [FN, TP]]。
    cm = confusion_matrix(y_test, y_pred_threshold)

    # 4. 通过行列索引取出 FP 和 FN。
    # FP 是把未流失客户误判为流失，FN 是漏掉真实流失客户。
    fp = cm[0, 1]
    fn = cm[1, 0]

    print(
        f"阈值={threshold:.2f}, "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FP={fp}, FN={fn}"
    )


阈值=0.50, Precision=0.657, Recall=0.559, F1=0.604, FP=109, FN=165
阈值=0.45, Precision=0.602, Recall=0.615, F1=0.608, FP=152, FN=144
阈值=0.40, Precision=0.568, Recall=0.668, F1=0.614, FP=190, FN=124
阈值=0.35, Precision=0.543, Recall=0.706, F1=0.614, FP=222, FN=110


## 5. 阈值选择结论

我选择 threshold = 0.40。

因为客户挽留更关注尽可能识别真实会流失的客户，
该阈值下 Recall 为 0.668，FN 为 124。
同时，我愿意接受 FP 为 190 带来的额外联系成本。

## 6. 保存、加载模型与新客户预测


In [12]:
import joblib

model_path = "churn_pipeline.joblib"

# 保存完整 Pipeline：文件同时包含预处理规则和逻辑回归模型，app.py 可直接加载
joblib.dump(pipeline, model_path)

# 重新加载模型，验证保存后的文件可以正常使用
loaded_pipeline = joblib.load(model_path)

print(type(loaded_pipeline))
print(loaded_pipeline.named_steps)



# 构造一条虚拟客户数据；字段名称和取值必须与训练数据保持一致
new_customer = pd.DataFrame([{
    "SeniorCitizen": 0,
    "tenure": 6,
    "MonthlyCharges": 85.00,
    "TotalCharges": 510.00,
    "gender": "Male",
    "Partner": "No",
    "Dependents": "No",
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check"
}])

print(new_customer.shape)
print(new_customer.columns.tolist())


# 使用加载后的完整 Pipeline 预测流失概率
# predict_proba(...)[0, 1] 表示取第 1 条客户属于类别 1（流失）的概率
new_customer_probability = loaded_pipeline.predict_proba(
    new_customer
)[0, 1]

print(f"新客户流失概率：{new_customer_probability:.2%}")

<class 'sklearn.pipeline.Pipeline'>
{'preprocessor': ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])]), 'classifier': LogisticRegression(max_iter=1000)}
(1, 19)
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetS

## 7. 环境版本记录


In [1]:
import pandas
import numpy
import scipy
import matplotlib
import sklearn
import joblib
import streamlit
import openai
import brotli

print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("streamlit:", streamlit.__version__)
print("openai:", openai.__version__)
print("Brotli:", brotli.__version__)

pandas: 2.2.3
numpy: 2.1.3
scipy: 1.15.1
matplotlib: 3.10.0
scikit-learn: 1.6.1
joblib: 1.4.2
streamlit: 1.45.1
openai: 3.11.0
Brotli: 1.2.0
